# Lexicon Results Exploration

Browse the uncertainty lexicon output: see which sentences were flagged,
inspect matches by category, and spot-check for false positives/negatives.

In [1]:
import sys
from pathlib import Path
import textwrap

import polars as pl

sys.path.insert(0, str(Path("..").resolve()))
from src.data_loading import load_full
from src.preprocessing import preprocess
from src.uncertainty_lexicon import analyze_text, get_category_names

RESULTS_DIR = Path("../results/tables")

## 1. Saved Lexicon Results

In [2]:
cat_rates = pl.read_csv(RESULTS_DIR / "lexicon_category_rates.csv")
examples = pl.read_csv(RESULTS_DIR / "lexicon_example_matches.csv")
trace_stats = pl.read_csv(RESULTS_DIR / "lexicon_trace_stats.csv")

print("Category detection rates (across 8,378 sentences from 500 traces):")
print(cat_rates)
print(f"\nExample matches saved: {len(examples)}")
print(f"Trace stats saved: {len(trace_stats)}")

Category detection rates (across 8,378 sentences from 500 traces):
shape: (6, 3)
┌──────────────────────┬──────────────────┬──────────────────┐
│ category             ┆ n_sentences_with ┆ pct_of_sentences │
│ ---                  ┆ ---              ┆ ---              │
│ str                  ┆ i64              ┆ f64              │
╞══════════════════════╪══════════════════╪══════════════════╡
│ epistemic_hedge      ┆ 187              ┆ 2.232            │
│ evidential_marker    ┆ 7                ┆ 0.084            │
│ explicit_uncertainty ┆ 5                ┆ 0.06             │
│ probability_language ┆ 85               ┆ 1.015            │
│ modal_hedge          ┆ 186              ┆ 2.22             │
│ approximator         ┆ 25               ┆ 0.298            │
└──────────────────────┴──────────────────┴──────────────────┘

Example matches saved: 100
Trace stats saved: 500


In [3]:
# Browse saved example matches (top 100 uncertain sentences by marker count)
# These are truncated to 200 chars in the CSV — we'll also show full text below
print(f"{'='*90}")
print(f"TOP UNCERTAIN SENTENCES (from saved results)")
print(f"{'='*90}")

for i in range(min(20, len(examples))):
    row = examples.row(i, named=True)
    cats = []
    for c in get_category_names():
        val = row.get(f"cat_{c}", 0)
        if val and val > 0:
            cats.append(f"{c}({val})")
    
    print(f"\n[{i}] markers={row['total_markers']}  categories: {', '.join(cats)}")
    print(f"    matched: {row['matched_phrases']}")
    text = row['text']
    # Wrap long text
    wrapped = textwrap.fill(text, width=88, initial_indent='    ', subsequent_indent='    ')
    print(wrapped)

TOP UNCERTAIN SENTENCES (from saved results)

[0] markers=7  categories: probability_language(1), modal_hedge(2), approximator(4)
    matched: likely; could; Could; around 0; around 5
    :  Blackjack: - Has one of the lowest house edges (around 0.5-1% with optimal play)
    - Requires some skill to play optimally - I could make small, strategic bets to try
    to recover my $20 loss  Poker: -

[1] markers=3  categories: epistemic_hedge(1), modal_hedge(2)
    matched: I think; could; might
    The chances of busting are zero, but standing could be a good option if I think the
    dealer might bust.

[2] markers=3  categories: epistemic_hedge(1), probability_language(1), modal_hedge(1)
    matched: maybe; probably; might
    For the growth chart, we probably just want: - The "Growth Tracker" sheet (the
    visual one with charts) - maybe 1-2 pages - Not the data sheets with 732 rows each
    We might need to: 1.

[3] markers=3  categories: epistemic_hedge(1), probability_language(1), ap

## 2. Live Analysis: Run Lexicon on Specific Rows

Load the preprocessed dataset and run the uncertainty lexicon on any row.

In [4]:
df = load_full().collect()
df = preprocess(df)
print(f"Preprocessed dataset: {len(df):,} rows")
print(f"Columns: {df.columns}")

Preprocessed dataset: 9,923 rows
Columns: ['messages', 'response', 'reasoning', 'model', 'tools', 'tool_calls', 'model_family', 'model_provider', 'user_prompt', 'n_turns', 'reasoning_len_chars', 'response_len_chars', 'prompt_len_chars', 'nsfw_flag']


In [5]:
def analyze_row(idx: int, show_all: bool = False, max_chars: int = 300) -> None:
    """Run the uncertainty lexicon on a specific row and display results.
    
    Args:
        idx: Row index in the preprocessed dataframe
        show_all: If True, show all sentences. If False, only uncertain ones.
        max_chars: Max chars per sentence to display (0 = no limit)
    """
    row = df.row(idx, named=True)
    reasoning = str(row.get("reasoning", ""))
    
    print(f"{'='*90}")
    print(f"ROW {idx}  |  model: {row['model']}  |  family: {row['model_family']}")
    print(f"reasoning: {len(reasoning):,} chars  |  nsfw: {row.get('nsfw_flag', '?')}")
    print(f"{'='*90}")
    
    # Show user prompt
    prompt = str(row.get("user_prompt", ""))
    if prompt:
        print(f"\nUSER PROMPT:")
        print(textwrap.fill(prompt[:500], width=88, initial_indent='  ', subsequent_indent='  '))
        if len(prompt) > 500:
            print(f"  ... [{len(prompt):,} chars total]")
    
    # Run lexicon
    results = analyze_text(reasoning, use_spacy=True)
    n_uncertain = sum(1 for r in results if r.has_uncertainty)
    
    print(f"\n{len(results)} sentences, {n_uncertain} uncertain ({n_uncertain/len(results)*100:.1f}%)")
    print(f"{'─'*90}")
    
    for i, r in enumerate(results):
        if not show_all and not r.has_uncertainty:
            continue
        
        tag = "UNCERTAIN" if r.has_uncertainty else "         "
        cats = {k: v for k, v in r.categories.items() if v > 0}
        
        text = r.text
        if max_chars and len(text) > max_chars:
            text = text[:max_chars] + "..."
        
        print(f"\n  [{i:3d}] {tag}")
        if r.has_uncertainty:
            print(f"         categories: {cats}")
            print(f"         matched: {r.matched_phrases}")
        wrapped = textwrap.fill(text, width=84, initial_indent='         ', subsequent_indent='         ')
        print(wrapped)


def find_uncertain_rows(min_rate: float = 0.1, max_results: int = 20) -> pl.DataFrame:
    """Find rows with high uncertainty rates in reasoning traces.
    
    Analyzes a sample and returns rows sorted by uncertainty rate.
    """
    records = []
    sample = df.head(500)  # analyze first 500 for speed
    for i in range(len(sample)):
        row = sample.row(i, named=True)
        reasoning = str(row.get("reasoning", ""))
        if len(reasoning) < 50:
            continue
        results = analyze_text(reasoning, use_spacy=True)
        n_uncertain = sum(1 for r in results if r.has_uncertainty)
        rate = n_uncertain / len(results) if results else 0
        if rate >= min_rate:
            records.append({
                "row_idx": i,
                "model_family": row["model_family"],
                "n_sentences": len(results),
                "n_uncertain": n_uncertain,
                "uncertainty_rate": round(rate, 3),
                "reasoning_len": len(reasoning),
            })
    
    result = pl.DataFrame(records).sort("uncertainty_rate", descending=True).head(max_results)
    print(f"Found {len(records)} rows with uncertainty rate >= {min_rate}")
    return result


def find_by_category(category: str, max_results: int = 10) -> list[dict]:
    """Find sentences matching a specific uncertainty category.
    
    Args:
        category: One of the category names from get_category_names()
        max_results: Max sentences to return
    """
    found = []
    for i in range(min(500, len(df))):
        row = df.row(i, named=True)
        reasoning = str(row.get("reasoning", ""))
        if len(reasoning) < 50:
            continue
        results = analyze_text(reasoning, use_spacy=True)
        for sent_idx, r in enumerate(results):
            if r.categories.get(category, 0) > 0:
                found.append({
                    "row_idx": i,
                    "sent_idx": sent_idx,
                    "model_family": row["model_family"],
                    "text": r.text[:200],
                    "matched": r.matched_phrases,
                    "count": r.categories[category],
                })
                if len(found) >= max_results:
                    return found
    return found

In [9]:
# Analyze a specific row — change idx to explore different rows
# show_all=True to see every sentence, False for uncertain-only
analyze_row(120, show_all=True)

ROW 120  |  model: minimax-m2  |  family: minimax
reasoning: 635 chars  |  nsfw: False

USER PROMPT:
  Game state: {   "phase": "betting",   "balance": 955 }  What is your decision?

7 sentences, 1 uncertain (14.3%)
──────────────────────────────────────────────────────────────────────────────────────────

  [  0]          
         I need to make a decision for the blackjack game.

  [  1]          
         The current state shows I'm in the betting phase with a balance of $955.

  [  2]          
         The minimum bet is $10 and the maximum is $5000.

  [  3]          
         Since I need to make a bet to proceed, I should choose an amount that's
         reasonable given my current balance.

  [  4]          
         I have $955 to work with, so I should bet something that allows me to play
         multiple hands while still being significant enough to potentially win
         money.

  [  5] UNCERTAIN
         categories: {'epistemic_hedge': 1}
         matched: ['I think']

In [10]:
# Find rows with the highest uncertainty rates
high_unc = find_uncertain_rows(min_rate=0.1)
high_unc

Found 71 rows with uncertainty rate >= 0.1


row_idx,model_family,n_sentences,n_uncertain,uncertainty_rate,reasoning_len
i64,str,i64,i64,f64,i64
154,"""other""",25,13,0.52,2709
99,"""claude-opus""",2,1,0.5,886
76,"""claude-opus""",5,2,0.4,668
277,"""deepseek""",10,4,0.4,457
282,"""gpt""",25,9,0.36,2991
…,…,…,…,…,…
495,"""other""",13,4,0.308,634
233,"""minimax""",17,5,0.294,2201
96,"""glm""",24,7,0.292,1898


In [11]:
# Inspect the row with the highest uncertainty rate
if len(high_unc) > 0:
    best_idx = high_unc["row_idx"][0]
    analyze_row(best_idx, show_all=False)

ROW 154  |  model: nemotron-nano-9b-v2:free  |  family: other
reasoning: 2,709 chars  |  nsfw: False

USER PROMPT:
  <additional_data> Below are some potentially helpful/relevant pieces of information
  for figuring out how to respond: <current_file> Path: .github/copilot-instructions.md
  Currently selected line: 394 Line 394 content: `` </current_file> </additional_data>
  <user_query> 我是自定义的cursor模型 然后对话后 最后就会出现Request ID:
  b4228f9f-1e04-4280-b2fd-a886f0eb3981 ConnectError: [invalid_argument] Error     at
  Zru.$endAiConnectTransportReportError (vscode-file://vscode-
  app/c:/Program%20Files/cursor/resources/app/out/v
  ... [1,868 chars total]

25 sentences, 13 uncertain (52.0%)
──────────────────────────────────────────────────────────────────────────────────────────

  [  2] UNCERTAIN
         categories: {'modal_hedge': 1}
         matched: ['might']
         Let me break down what might be going on here.

  [  8] UNCERTAIN
         categories: {'epistemic_hedge': 1}
         ma

## 3. Explore by Category

Categories: `epistemic_hedge`, `evidential_marker`, `explicit_uncertainty`, `probability_language`, `modal_hedge`, `approximator`

In [12]:
# Epistemic hedges: "I think", "perhaps", "maybe"
results = find_by_category("epistemic_hedge", max_results=10)
print(f"Found {len(results)} sentences with epistemic hedges:\n")
for r in results:
    print(f"  [{r['row_idx']}:{r['sent_idx']}] {r['model_family']}  matched={r['matched']}")
    print(textwrap.fill(r['text'], width=84, initial_indent='    ', subsequent_indent='    '))
    print()

Found 10 sentences with epistemic hedges:

  [2:20] glm  matched=['Maybe']
    Maybe "Spurts" or "Suddenly" or "Sludge".

  [2:32] glm  matched=['maybe']
    4. **Location Infoboard**: Route 9, afternoon, specific date (I'll pick a random
    date fitting the Pokemon world, maybe month 10 day 5).

  [2:46] glm  matched=['maybe']
    Her reaction: Eyes wide, mouth open (maybe), sputtering.

  [10:20] gemini  matched=['maybe']
    maybe).    - "No shoes" rule established.

  [12:23] glm  matched=['Maybe']
    Maybe use "About Us" or just "About Us page".

  [12:50] glm  matched=['maybe']
    Since the input is in Indonesian, maybe the summary should be in Indonesian?

  [21:11] claude-opus  matched=['Maybe', 'maybe']
    - Activity tracking - Profile/Stats - Maybe a workout or challenges section
    I'll use: - Expo Linear Gradient for beautiful gradients - Lucide icons for
    clean icons - Modern color scheme (think energe

  [24:11] claude-opus  matched=['Maybe']
    Maybe introduce m

In [13]:
# Evidential markers: "it seems", "it appears", "apparently" (rare — 0.08%)
results = find_by_category("evidential_marker", max_results=10)
print(f"Found {len(results)} sentences with evidential markers:\n")
for r in results:
    print(f"  [{r['row_idx']}:{r['sent_idx']}] {r['model_family']}  matched={r['matched']}")
    print(textwrap.fill(r['text'], width=84, initial_indent='    ', subsequent_indent='    '))
    print()

Found 3 sentences with evidential markers:

  [2:75] glm  matched=['seemingly']
    It is nothing like a human's pathetic load; this sludge is heavy, chunky, and
    seemingly endless.

  [153:31] minimax  matched=['It looks like']
    It looks like we're stuck in a greeting loop.

  [191:2] claude-opus  matched=['maybe', 'maybe', 'apparently', 'roughly']
    Used to be responsible big brother, now feminized slut. - Logan: 9ft badger,
    massive muscles, 16-inch cock, deep voice, bratty, vulgar, possessive but
    apparently okay with Chris getting fucked by othe



In [ ]:
# Explicit uncertainty: "I'm not sure", "I may be mistaken" (rarest — 0.06%)
results = find_by_category("explicit_uncertainty", max_results=10)
print(f"Found {len(results)} sentences with explicit uncertainty:\n")
for r in results:
    print(f"  [{r['row_idx']}:{r['sent_idx']}] {r['model_family']}  matched={r['matched']}")
    print(textwrap.fill(r['text'], width=84, initial_indent='    ', subsequent_indent='    '))
    print()

In [ ]:
# Modal hedges: "might", "could" (POS-verified as auxiliaries)
results = find_by_category("modal_hedge", max_results=10)
print(f"Found {len(results)} sentences with modal hedges:\n")
for r in results:
    print(f"  [{r['row_idx']}:{r['sent_idx']}] {r['model_family']}  matched={r['matched']}")
    print(textwrap.fill(r['text'], width=84, initial_indent='    ', subsequent_indent='    '))
    print()

## 4. Browse by Model Family

In [ ]:
# Which model families are in the preprocessed data?
print(df.group_by("model_family").len().sort("len", descending=True))

In [ ]:
# Pick a model family and analyze a row from it
FAMILY = "deepseek"  # try: "glm", "gpt", "gemini", "minimax", "claude-opus", "qwen"

family_indices = [
    i for i in range(len(df))
    if df.row(i, named=True)["model_family"] == FAMILY
]
print(f"{len(family_indices)} rows from '{FAMILY}'")

if family_indices:
    analyze_row(family_indices[0], show_all=False)

## 5. Sandbox

```python
# Analyze any row
analyze_row(42, show_all=True)       # show all sentences
analyze_row(42, show_all=False)      # uncertain sentences only
analyze_row(42, max_chars=0)         # no truncation

# Find rows with high uncertainty
high_unc = find_uncertain_rows(min_rate=0.15)
analyze_row(high_unc["row_idx"][0])

# Find sentences by category
find_by_category("approximator", max_results=5)

# Run lexicon on arbitrary text
results = analyze_text("I think this might work, but I'm not entirely sure.")
for r in results:
    print(r)
```

In [ ]:
# Sandbox — try anything here
